# 🤠 Ranch Lab — a hands-on tour of the loop

This notebook walks every component of the Ranch harness, bottom-up: the DB, the run
lifecycle, the Foreman's decision functions, the checkpoint gate (old blocking vs. new
one-shot), and finally a **real live run** against the `toy` agent.

**Safety:** everything here runs against an isolated sandbox (`~/.ranch-sandbox`), never
your real `~/.ranch`. The live section operates only on the throwaway `toybox` repo.

**Kernel:** use the `Ranch (sandbox)` kernel (the ranch venv). If you don't see it,
run in a terminal:
```bash
cd ~/code/citemed/ranch && .venv/bin/python -m ipykernel install --user --name ranch-sandbox --display-name "Ranch (sandbox)"
```

Companion reading: `docs/foreman.md` (the architecture we're migrating to) and
`ARCHITECTURE.md` (the current system).

## 0. Setup — point at the sandbox

`RANCH_HOME` **must** be set *before* importing anything from `ranch`, because
`ranch/config.py` reads it at import time to build `DATABASE_URL`.
If you change it later, **restart the kernel**.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

os.environ["RANCH_HOME"] = str(Path.home() / ".ranch-sandbox")   # ← the isolation switch
RANCH  = Path.home() / "code/citemed/ranch"
TOYBOX = Path.home() / "code/citemed/toybox"
sys.path.insert(0, str(RANCH))

from ranch.db import init_db, db_session
from ranch import config
init_db()   # create tables + run the additive column migrations

print("RANCH_HOME :", config.RANCH_HOME)
print("DB         :", config.DATABASE_URL)
print("agents     :", {k: str(v.worktree) for k, v in config.reload_agents().items()})

In [ ]:
# Small helper: run the ranch CLI with the sandbox env, print the output.
def ranch_cli(*args: str) -> str:
    r = subprocess.run(
        [str(RANCH / ".venv/bin/ranch"), *args],
        capture_output=True, text=True, env={**os.environ},
    )
    out = (r.stdout or "") + (r.stderr or "")
    print(out)
    return out

_ = ranch_cli("runs")   # empty table on a fresh sandbox

## 1. The cast (60-second architecture)

| Component | File | Role |
|---|---|---|
| **RanchHand** (→ the Foreman) | `ranch/hand.py` | Daemon: polls, decides, dispatches sessions |
| **Orchestrator** | `ranch/runner/orchestrator.py` | Wraps ONE Claude SDK session |
| **ranch MCP tools** | `ranch/runner/tools.py` | `record_checkpoint`, `record_state`, `run_acceptance`, … |
| **PostToolUse hooks** | `ranch/runner/checkpoints.py` etc. | Where tool behavior actually lives |
| **DB** | `$RANCH_HOME/ranch.db` | `Run`, `Checkpoint`, `Dossier`, `Interjection` |

**The core trick:** an MCP tool body only *acknowledges*. The PostToolUse **hook** that
fires afterwards does the real work — persisting rows, and (for gated checkpoints)
deciding whether the agent may continue. The hook can attach text to the tool result
(`additionalContext`), which is how a human decision reaches the agent.

## 2. The `Run` row — one session's lifecycle

Every Claude session gets a `Run` row. Its `state` walks:
`queued → planning → in_development → needs_approval → … → completed | stopped | error`.

Let's create one by hand and look at it.

In [ ]:
from ranch.models import Run, Dossier, Checkpoint, Interjection
from datetime import datetime, timezone

with db_session() as db:
    run = Run(agent="toy", ticket="TOY-DEMO", cwd=str(TOYBOX),
              initial_prompt="(hand-seeded demo row)", state="planning")
    db.add(run); db.flush()
    demo_run_id = run.id

print("created run #", demo_run_id)
_ = ranch_cli("runs")

## 3. Dossier + Checkpoint — how the agent reports back

While a session runs, the agent calls MCP tools and the hooks persist:

- **`record_state` → `Dossier` row** — a structured self-report (plan, `just_did`,
  state, blockers, acceptance criteria). This is what the cockpit renders.
- **`record_checkpoint` → `Checkpoint` row** — a workflow gate
  (`plan_ready`, `tests_green`, `pre_push`). Kinds in `APPROVAL_REQUIRED` pause the run.

Let's fake what an agent would write, then read it back like the UI does.

In [ ]:
payload = {
    "plan": [{"step": "write add()", "status": "done"},
              {"step": "write tests", "status": "in_progress"}],
    "just_did": "Implemented add(); writing tests next",
    "state": "coding",
    "files_touched": ["toybox/mathx.py"],
}
with db_session() as db:
    db.add(Dossier(run_id=demo_run_id, state="coding", payload_json=json.dumps(payload)))
    db.add(Checkpoint(run_id=demo_run_id, kind="plan_ready", summary="Plan: mathx.py with add() + tests"))

_ = ranch_cli("dossier", str(demo_run_id))

## 4. The Foreman's decision functions

`RanchHand.run()` (hand.py:813) is a poll loop. Each tick it asks, in order:

1. **`_active_run_for(agent)`** — anything in flight? → leave it alone
2. **`_find_approved_parked_propose(agent)`** — operator approved a plan? → fire execute
3. **`_last_parked_run_for(agent)`** — something parked awaiting review? → idle
4. otherwise → triage Jira → queue candidates → operator kicks off → propose

These are plain functions over the DB — we can drive the whole decision tree by
seeding rows. **This is exactly how the hermetic tests work** (`tests/test_hand.py`).

In [ ]:
from ranch.hand import _active_run_for, _find_approved_parked_propose, _last_parked_run_for

# Tick 1: our demo run is state='planning' (non-terminal) → the hand sees it as ACTIVE
active = _active_run_for("toy")
print("active run:", active.id if active else None, "→ the hand would idle this tick")

In [ ]:
# Now simulate: the propose session finished and PARKED with a plan awaiting review.
with db_session() as db:
    r = db.query(Run).filter_by(id=demo_run_id).one()
    r.state, r.exit_reason = "completed", "completed"
    r.ended_at = datetime.now(timezone.utc)
    db.add(Dossier(run_id=demo_run_id, state="parked", payload_json=json.dumps({
        "plan": [{"step": "write add()", "status": "pending"}],
        "just_did": "Proposal ready for operator review",
        "state": "parked",
        "acceptance": [{"kind": "script", "name": "pytest green",
                         "cmd": "python3 -m pytest -q", "pass_pattern": "passed"}],
        "details": "Add mathx.py with add(a,b); TDD.",
    })))

print("active now:", _active_run_for("toy"))
parked = _last_parked_run_for("toy")
print("parked    :", parked.id if parked else None, "→ hand idles, waiting for YOU")

In [ ]:
# YOU approve (this is what `ranch approve <id>` writes: an Interjection row)…
_ = ranch_cli("approve", str(demo_run_id))

# …and on its next tick the hand's finder consumes the approval atomically:
approved = _find_approved_parked_propose("toy")
print("\napproved propose found:", approved.propose_run.ticket if approved else None)
print("carried payload keys  :", list(approved.parked_payload.keys()) if approved else None)

# Run it again — the interjection was consumed, so it can't double-fire:
print("second call           :", _find_approved_parked_propose("toy"))

That's the propose→execute handoff: the approved plan + acceptance contract get carried
onto a fresh execute `Run` (`_create_execute_run`), so the coding session starts with
the vetted plan and `run_acceptance` finds its checks without re-deriving anything.

## 5. THE GATE — old blocking vs. new one-shot ⭐

This is the heart of the Foreman migration (`docs/foreman.md §5`).

**OLD (still the live default):** when the agent calls `record_checkpoint(pre_push)`,
the PostToolUse hook literally `await`s a human decision — the agent's tool call is
frozen mid-flight, holding the whole SDK process hostage until someone approves.

**NEW (`one_shot=True`, landed but not yet wired into the hand):** the session records
the checkpoint, tells the agent *"PAUSED — do not proceed"*, and **exits cleanly**.
The run keeps `exit_reason='paused_at_gate'` and its `sdk_session_id`, so the Foreman
can resume it later with the decision injected. Approval becomes a graph edge, not a
parked thread.

Both paths are demonstrable **hermetically** — no tokens, no SDK process.

In [ ]:
from ranch.runner.orchestrator import Orchestrator

# ---- OLD path: the orchestrator arms an in-session wait ----
with db_session() as db:
    r = Run(agent="toy", ticket="TOY-OLD", cwd="/tmp", initial_prompt="x", state="planning")
    db.add(r); db.flush(); old_id = r.id

old = Orchestrator("toy", Path("/tmp"), "TOY-OLD", "brief")   # one_shot defaults OFF
old.run_id = old_id
await old.on_checkpoint("pre_push", "Diff ready — may I push?", None)

print("awaiting_approval:", old._awaiting_approval)
print("event set?       :", old._approval_ready.is_set(),
      "← unset = the hook WOULD now block until approve/reject arrives")

In [ ]:
# ---- NEW path: one_shot exits at the gate instead ----
from ranch.runner.checkpoints import make_checkpoint_hook, CHECKPOINT_TOOL

with db_session() as db:
    r = Run(agent="toy", ticket="TOY-NEW", cwd="/tmp", initial_prompt="x", state="planning")
    db.add(r); db.flush(); new_id = r.id

new = Orchestrator("toy", Path("/tmp"), "TOY-NEW", "brief", one_shot=True)
new.run_id = new_id

# Simulate the agent's tool call arriving at the hook:
hook = make_checkpoint_hook(new).hooks[0]
result = await hook({"tool_name": CHECKPOINT_TOOL,
                      "tool_input": {"kind": "pre_push", "summary": "Diff ready"}},
                     "tool-use-1", None)

print("paused_at_gate :", new._paused_at_gate)
print("stop_requested :", new.stop_requested, "← main loop will exit cleanly")
print("\nwhat the AGENT sees on its tool result:\n ",
      result["hookSpecificOutput"]["additionalContext"])

In [ ]:
# The session then finalizes — NOT terminal-complete, but resumable:
await new._finalize()

with db_session() as db:
    r = db.query(Run).filter_by(id=new_id).one()
    print("exit_reason:", r.exit_reason, "| state:", r.state)

# And when the operator decides, the Foreman resumes the SAME session, injecting
# the decision as the first message. This is the exact text the agent receives:
from ranch.runner.messages import HumanDecision
print("\n--- resume message (approved) ---")
print(HumanDecision(checkpoint_kind="pre_push", decision="approved", ticket="TOY-NEW").to_prompt()[:400], "…")

**Why this matters:** in the OLD path the UI must babysit a live, blocked process —
the root of the Electron pain. In the NEW path all state is in the DB; the UI becomes a
read-view, async events (PR comments, CI) can wake runs, and `claude --resume` takeover
(Flavor A) gets a clean seam.

> Status: mechanism landed (default off) · Foreman wiring = next slice. Resumed runs
> stay one-shot, so a later gate pauses the session again the same way.

## 6. Interjections — the operator's channel

Every operator action (`approve` / `reject` / `note` / `stop`) is just an
**`Interjection` row** with `processed_at=NULL`. Whoever is responsible consumes it:
a live orchestrator's 500 ms poll loop (old path) or the hand's finders (new path).
This means the CLI, the web cockpit, and future Slack commands all share one channel —
they just INSERT a row.

In [ ]:
_ = ranch_cli("note", str(demo_run_id), "prefer double quotes in tests please")
with db_session() as db:
    for i in db.query(Interjection).filter_by(run_id=demo_run_id).all():
        print(f"#{i.id} kind={i.kind!r:10} content={i.content!r:45} processed={i.processed_at}")

## 7. 🔴 LIVE — dispatch a real run on the toy agent

Everything above was free. This section spawns a **real Claude Code session** against
`~/code/citemed/toybox` (a throwaway repo with a local bare `origin`, so the push step
works for real). Cost: one coding session (cents). Requires `claude` CLI auth.

The structured workflow will: plan → **gate: `plan_ready` (you approve)** → TDD →
self-verify → **gate: `pre_push` (you approve)** → branch + push.

Flip `LIVE = True` to arm the cells.

In [ ]:
LIVE = False   # ← flip to True to spend real tokens

BRIEF = (
    "Create toybox/mathx.py with add(a, b) and multiply(a, b), and tests in "
    "tests/test_mathx.py covering both (include a negative-number case). "
    "Run `python3 -m pytest -q` and iterate until green. There is NO PR platform: "
    "after pushing your branch, do not attempt bb/gh pr create — just stop."
)

if LIVE:
    out = ranch_cli("dispatch", "toy", "--ticket", "TOY-1", "--brief", BRIEF)
else:
    print("LIVE is False — flip it to dispatch for real.")

In [ ]:
# Watch it (re-run this cell as it progresses). Find your run id in `ranch runs`.
if LIVE:
    _ = ranch_cli("runs")
    # RUN_ID = 123                          # ← set from the table above, then:
    # _ = ranch_cli("dossier", str(RUN_ID))  # agent's structured self-report
    # _ = ranch_cli("status", str(RUN_ID))   # checkpoints + pending gates
    # _ = ranch_cli("log", str(RUN_ID))      # path to the live log — tail it in a terminal
else:
    print("LIVE is False.")

In [ ]:
# When it parks at a gate (state → needs_approval), approve it — twice over the run:
# once for plan_ready, later for pre_push.
if LIVE:
    pass
    # _ = ranch_cli("approve", str(RUN_ID))
    # …or push back instead and watch it revise:
    # _ = ranch_cli("reject", str(RUN_ID), "--reason", "also add a divide() with zero-check")
else:
    print("LIVE is False.")

In [ ]:
# Afterwards: did the Hand actually write code, test it, and push?
if LIVE:
    print(subprocess.run(["git", "-C", str(TOYBOX), "log", "--oneline", "--all", "-5"],
                          capture_output=True, text=True).stdout)
    print(subprocess.run(["git", "-C", str(TOYBOX / "..") , "-C", str(Path.home() / "code/citemed/toybox-origin.git"),
                          "branch", "-a"], capture_output=True, text=True).stdout)
    t = TOYBOX / "tests/test_mathx.py"
    print(t.read_text() if t.exists() else "(test file not created yet)")
else:
    print("LIVE is False.")

## 8. Where this is going

You just drove, by hand, exactly what the **Foreman** automates:
*find work → propose → gate → execute → gate → push*, with you as the gate.

The migration plan (`docs/foreman.md §14`):
1. **One-shot de-block** — mechanism ✅ landed (you demoed it in §5); hand wiring next
2. **Brand Inspector** — independent evaluator replaces the self-judge (the paper's
   "nodding loop" fix); ideally a different vendor (Codex) behind the adapter seam
3. **Test rig** — Playwright MCP + deterministic auth seeding
4. **Cockpit** — the read-view web app (`ranch serve` + `console/`), Electron is gone
5. **Design skill + Design Inspector** — Tailwind prototypes, taste gates

Useful commands while exploring:
```bash
RANCH_HOME=~/.ranch-sandbox .venv/bin/ranch runs          # list runs
RANCH_HOME=~/.ranch-sandbox .venv/bin/ranch hand start toy # tier-3: the full daemon
RANCH_HOME=~/.ranch-sandbox .venv/bin/ranch serve          # the web cockpit sidecar
.venv/bin/python -m pytest tests/test_orchestrator_loop.py -q   # the gate's test suite
```

To reset the sandbox completely: `rm -rf ~/.ranch-sandbox` and re-run cell 0
(you may want to re-create `config.toml` — see the repo's `notebooks/` README or
re-register the toy agent by hand).